# PPE Safety Detection with YOLO

Interactive notebook for the YOLO PPE (Personal Protective Equipment) detection pipeline:

1. **YOLO Health** — Query the YOLO serving endpoint
2. **PPE Detection** — Run inference and visualize bounding boxes
3. **Kafka PPE Events** — Consume from `cv.ppe.detections` topic
4. **Granite LLM Analysis** — Get natural language safety reports
5. **Compliance Dashboard** — Plot PPE compliance charts

In [ ]:
!pip install -q requests numpy opencv-python-headless matplotlib kafka-python Pillow

In [ ]:
import requests
import numpy as np
import json
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from io import BytesIO
import time
from collections import Counter

YOLO_URL = "http://yolo-ppe-serving.neuroface.svc:8000"
GRANITE_URL = "http://granite-llm-predictor.neuroface.svc:8080"
KAFKA_BOOTSTRAP = "cdc-cluster-kafka-bootstrap.kafka-cdc.svc:9092"
KAFKA_TOPIC = "cv.ppe.detections"

EXPECTED_PPE = ["hardhat", "safety-vest", "goggles"]

print("Configuration loaded")
print(f"  YOLO:    {YOLO_URL}")
print(f"  Granite: {GRANITE_URL}")
print(f"  Kafka:   {KAFKA_BOOTSTRAP}")
print(f"  Topic:   {KAFKA_TOPIC}")

## 1. YOLO Health Check

Query `/health` to verify the YOLO serving endpoint is running and the model is loaded.

In [ ]:
try:
    resp = requests.get(f"{YOLO_URL}/health", timeout=10)
    health = resp.json()
    print("YOLO Status:", resp.status_code)
    print(json.dumps(health, indent=2))

    resp2 = requests.get(f"{YOLO_URL}/v1/models", timeout=10)
    models = resp2.json()
    print(f"\nModel: {models.get('model')}")
    print(f"Classes: {len(models.get('classes', {}))}")
    print("\nCOCO classes relevant to PPE:")
    for cid, name in models.get("classes", {}).items():
        if name in ["person", "tie", "backpack", "handbag", "umbrella", "hat"]:
            print(f"  [{cid}] {name}")
except requests.exceptions.ConnectionError:
    print("ERROR: Cannot reach YOLO serving. Is yolo-ppe-serving running in neuroface namespace?")

## 2. PPE Detection Inference

Send a test image to the YOLO endpoint and visualize detections with bounding boxes.

In [ ]:
TEST_IMAGE_URL = "https://images.unsplash.com/photo-1504307651254-35680f356dfd?w=640"

detections = []
original_img = None

try:
    img_resp = requests.get(TEST_IMAGE_URL, timeout=15)
    original_img = Image.open(BytesIO(img_resp.content))
    print(f"Image downloaded: {original_img.size[0]}x{original_img.size[1]}px")

    print("\nRunning YOLO inference...")
    result = requests.post(
        f"{YOLO_URL}/v1/predict",
        json={"image_url": TEST_IMAGE_URL, "confidence": "0.3"},
        timeout=30
    ).json()

    detections = result.get("detections", [])
    print(f"Detected {len(detections)} object(s)")
    for i, det in enumerate(detections):
        print(f"  [{i+1}] {det['class_name']} — conf={det['confidence']:.1%} bbox={det['bbox']}")

except requests.exceptions.ConnectionError:
    print("ERROR: Cannot reach YOLO serving or image URL")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

if original_img is not None and len(detections) > 0:
    ax.imshow(original_img)
    colors = {
        "person": "#FF4444",
        "hardhat": "#44FF44",
        "safety-vest": "#FFAA00",
        "goggles": "#4444FF",
    }
    for det in detections:
        bbox = det["bbox"]
        w = bbox["x2"] - bbox["x1"]
        h = bbox["y2"] - bbox["y1"]
        color = colors.get(det["class_name"], "#FFFFFF")
        rect = patches.Rectangle(
            (bbox["x1"], bbox["y1"]), w, h,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            bbox["x1"], bbox["y1"] - 5,
            f"{det['class_name']} {det['confidence']:.0%}",
            color=color, fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7)
        )
    ax.set_title(f"YOLO PPE Detection — {len(detections)} object(s)", fontsize=14)
else:
    ax.text(0.5, 0.5, "No image or no detections", ha="center", va="center", fontsize=14)
    ax.set_title("PPE Detection")
ax.axis("off")
plt.tight_layout()
plt.show()

## 3. PPE Compliance Analysis

Check detected objects against the expected PPE list to determine compliance status.

In [ ]:
detected_classes = {d["class_name"].lower() for d in detections}
person_count = sum(1 for d in detections if d["class_name"] == "person")
present_ppe = [c for c in EXPECTED_PPE if c.lower() in detected_classes]
missing_ppe = [c for c in EXPECTED_PPE if c.lower() not in detected_classes]

if person_count == 0:
    status = "NO PERSONS DETECTED"
    status_color = "gray"
elif not missing_ppe:
    status = "COMPLIANT"
    status_color = "green"
else:
    status = "VIOLATION"
    status_color = "red"

print(f"PPE Status: {status}")
print(f"Persons detected: {person_count}")
print(f"Present PPE: {present_ppe or 'None'}")
print(f"Missing PPE: {missing_ppe or 'None'}")
print(f"All detected classes: {sorted(detected_classes)}")

## 4. Granite LLM Safety Analysis

Send detection results to the Granite 3.1 2B LLM for a natural language safety compliance report.

In [ ]:
detected_objects = [d["class_name"] for d in detections]
prompt = (
    f"You are a workplace safety AI assistant. Analyze these YOLO detections from a factory floor camera:\n"
    f"- Persons detected: {person_count}\n"
    f"- Objects detected: {', '.join(detected_objects)}\n"
    f"- Expected PPE: {', '.join(EXPECTED_PPE)}\n\n"
    f"Determine if each operator is in compliance with safety protocol. "
    f"List any missing PPE and explain the risk in 2-3 sentences."
)

try:
    llm_resp = requests.post(
        f"{GRANITE_URL}/v1/chat/completions",
        json={
            "model": "granite-3.1-2b-instruct",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 256,
            "temperature": 0.3,
        },
        timeout=30
    )
    llm_resp.raise_for_status()
    analysis = llm_resp.json()["choices"][0]["message"]["content"]
    print("Granite LLM Safety Analysis")
    print("=" * 40)
    print(analysis)
except requests.exceptions.ConnectionError:
    print("ERROR: Cannot reach Granite LLM. Is granite-llm running in neuroface namespace?")
    analysis = None
except Exception as e:
    print(f"LLM Error: {e}")
    analysis = None

## 5. Kafka PPE Events — `cv.ppe.detections`

Consume recent PPE compliance events from the Kafka topic.

In [ ]:
from kafka import KafkaConsumer
from kafka.errors import NoBrokersAvailable

events = []
try:
    consumer = KafkaConsumer(
        KAFKA_TOPIC,
        bootstrap_servers=KAFKA_BOOTSTRAP,
        auto_offset_reset="earliest",
        consumer_timeout_ms=10000,
        value_deserializer=lambda m: json.loads(m.decode("utf-8"))
    )
    print(f"Connected to {KAFKA_BOOTSTRAP}")
    print(f"Consuming from '{KAFKA_TOPIC}' (10s window)...\n")

    for msg in consumer:
        event = msg.value
        events.append(event)
        status = event.get("ppe_status", "unknown")
        ts = event.get("timestamp", "")
        persons = event.get("person_count", 0)
        missing = event.get("missing_ppe", [])
        icon = {"compliant": "OK", "violation": "!!"}.get(status, "--")
        print(f"[{ts}] {icon} status={status} persons={persons} missing={missing}")

    consumer.close()
    print(f"\nTotal events consumed: {len(events)}")

except NoBrokersAvailable:
    print("ERROR: Cannot connect to Kafka. Check SASL/SSL credentials if using secure connection.")
    print("Tip: For SASL_SSL, configure security_protocol and sasl_mechanism in KafkaConsumer.")

## 6. Compliance Dashboard

Visualize PPE compliance data from Kafka events.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top-left: PPE compliance distribution (pie chart)
ax1 = axes[0][0]
if events:
    statuses = Counter(e.get("ppe_status", "unknown") for e in events)
    colors_map = {"compliant": "#4CAF50", "violation": "#F44336", "no_persons": "#9E9E9E"}
    labels = list(statuses.keys())
    sizes = list(statuses.values())
    pie_colors = [colors_map.get(l, "#666") for l in labels]
    ax1.pie(sizes, labels=labels, colors=pie_colors, autopct="%1.0f%%",
            startangle=90, textprops={"fontsize": 11})
    ax1.set_title("PPE Compliance Distribution", fontsize=13, fontweight="bold")
else:
    ax1.text(0.5, 0.5, "No events", ha="center", va="center", fontsize=14)
    ax1.set_title("PPE Compliance Distribution")

# Top-right: Missing PPE frequency (bar chart)
ax2 = axes[0][1]
if events:
    all_missing = []
    for e in events:
        all_missing.extend(e.get("missing_ppe", []))
    if all_missing:
        missing_counts = Counter(all_missing)
        bars = ax2.bar(missing_counts.keys(), missing_counts.values(), color="#F44336")
        ax2.set_ylabel("Count")
        for bar, count in zip(bars, missing_counts.values()):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     str(count), ha="center", fontweight="bold")
    else:
        ax2.text(0.5, 0.5, "No missing PPE", ha="center", va="center", fontsize=14, color="green")
    ax2.set_title("Missing PPE Frequency", fontsize=13, fontweight="bold")
else:
    ax2.text(0.5, 0.5, "No events", ha="center", va="center", fontsize=14)
    ax2.set_title("Missing PPE Frequency")

# Bottom-left: Person count over time
ax3 = axes[1][0]
if events:
    person_counts = [e.get("person_count", 0) for e in events]
    ax3.plot(range(len(person_counts)), person_counts, "b-o", markersize=4, linewidth=1.5)
    ax3.fill_between(range(len(person_counts)), person_counts, alpha=0.2)
    ax3.set_xlabel("Event #")
    ax3.set_ylabel("Persons")
    ax3.set_title("Persons Detected Over Time", fontsize=13, fontweight="bold")
    ax3.grid(True, alpha=0.3)
else:
    ax3.text(0.5, 0.5, "No events", ha="center", va="center", fontsize=14)
    ax3.set_title("Persons Detected Over Time")

# Bottom-right: Detection class distribution
ax4 = axes[1][1]
if events:
    all_classes = []
    for e in events:
        for d in e.get("detections", []):
            all_classes.append(d.get("class_name", "unknown"))
    if all_classes:
        class_counts = Counter(all_classes).most_common(10)
        names, counts = zip(*class_counts)
        y_pos = range(len(names))
        ax4.barh(y_pos, counts, color="#2196F3")
        ax4.set_yticks(y_pos)
        ax4.set_yticklabels(names)
        ax4.invert_yaxis()
        ax4.set_xlabel("Count")
    else:
        ax4.text(0.5, 0.5, "No detections", ha="center", va="center", fontsize=14)
    ax4.set_title("YOLO Class Distribution (Top 10)", fontsize=13, fontweight="bold")
else:
    ax4.text(0.5, 0.5, "No events", ha="center", va="center", fontsize=14)
    ax4.set_title("YOLO Class Distribution")

plt.suptitle("PPE Safety Compliance Dashboard", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Pipeline Summary

| Component | Endpoint | Status |
|-----------|----------|--------|
| YOLO Serving | `yolo-ppe-serving.neuroface.svc:8000` | Check cell 1 |
| Granite LLM | `granite-llm-predictor.neuroface.svc:8080` | Check cell 4 |
| Kafka Topic | `cv.ppe.detections` on `cdc-cluster` | Check cell 5 |
| PPE Processor | `ppe-yolo-processor` in `kafka-cdc` | Polls YOLO -> Kafka -> Mailpit |
| OVMS (face) | `neuroface-ovms.neuroface.svc:8000` | Separate pipeline (untouched) |